# 05 Motion Planning With RRT

IK 只回答目标是否可达，planning 回答路径是否可行。

![RRT motion planning visual map](assets/figures/05_motion_planning_rrt.png)

## Learning Objectives

- Separate goal feasibility from path feasibility.
- Describe how RRT grows a collision-free tree through sampling.
- Relate planner parameters such as step size and max iterations to success rate.

## Checkpoint

- Run the RRT example and verify that the returned path is collision-free.
- Explain why the first and last path points matter.
- Identify one obstacle change that makes the planning problem harder.

## Practice Task

Increase the obstacle radius in small steps and record the first setting where planning becomes unreliable.

## Concept Map

![Colab concept image](assets/colab/05_motion_planning_rrt_concept.png)

**Concept image.** RRT grows a collision-free search tree and extracts a path once it connects start to goal.

In [ ]:
from pathlib import Path
import sys

COLAB_REPO_URL = "https://github.com/Hollis36/robotic-manipulation-learning.git"

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    in_colab = "google.colab" in sys.modules
    if in_colab:
        import subprocess

        PROJECT_ROOT = Path("/content/robotic-manipulation-learning")
        if not PROJECT_ROOT.exists():
            # Equivalent command: git clone --depth 1 <repo> <target>
            subprocess.run(["git", "clone", "--depth", "1", COLAB_REPO_URL, str(PROJECT_ROOT)], check=True)
    else:
        PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

PROJECT_ROOT


In [ ]:
import numpy as np
from rml.rrt import path_is_collision_free, rrt

start = np.array([0.0, 0.0])
goal = np.array([1.0, 1.0])
obstacles = [(0.5, 0.5, 0.2)]
path = np.array(rrt(start, goal, bounds=((0.0, 1.2), (0.0, 1.2)), obstacles=obstacles, step_size=0.15, max_iter=500, seed=4))
len(path), path_is_collision_free(path, obstacles), np.round(path[0], 3), np.round(path[-1], 3)

## Result Figure

Draw the obstacle, planned waypoints, start, and goal to verify the path visually.

The figure below is generated from the values computed in this notebook. Treat it as evidence from the code, not as a decorative illustration.

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({
    "figure.figsize": (7, 4.2),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})
fig, ax = plt.subplots()
for ox, oy, radius in obstacles:
    ax.add_patch(plt.Circle((ox, oy), radius, color='#E45756', alpha=0.35, label='obstacle'))
ax.plot(path[:, 0], path[:, 1], '-o', lw=3, label='RRT path')
ax.scatter([start[0]], [start[1]], s=120, marker='s', label='start')
ax.scatter([goal[0]], [goal[1]], s=160, marker='*', label='goal')
ax.set_xlim(-0.05, 1.25)
ax.set_ylim(-0.05, 1.25)
ax.set_aspect('equal', adjustable='box')
ax.set_xlabel('x position')
ax.set_ylabel('y position')
ax.legend(frameon=False)
plt.show()

## Parameter Experiment

The next cell is marked with `COLAB_PARAMETER_EXPERIMENT` so it is easy to find in Colab. Change `obstacle_radius` and print whether the deterministic planner finds a path and how long it is.

In [ ]:
# COLAB_PARAMETER_EXPERIMENT
for obstacle_radius in [0.12, 0.2, 0.28]:
    trial_obstacles = [(0.5, 0.5, obstacle_radius)]
    try:
        trial_path = np.array(rrt(start, goal, bounds=((0.0, 1.2), (0.0, 1.2)), obstacles=trial_obstacles, step_size=0.15, max_iter=500, seed=4))
        path_length = float(np.sum(np.linalg.norm(np.diff(trial_path, axis=0), axis=1)))
        print('radius=', obstacle_radius, 'waypoints=', len(trial_path), 'path_length=', round(path_length, 3))
    except RuntimeError:
        print('radius=', obstacle_radius, 'failure')

## Reflection Prompt

RRT 找到路径时，为什么还要检查每一段 path 是否 collision-free？结合采样、步长和障碍物半径解释。

Exercise: 增大 obstacle radius，观察何时 RRT 更难找到路径。